#### 1. Librerías.

In [1]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [2]:
#a. Modo de ejecución (se define en ./constantes/modo.txt, un solo lugar para los 4 notebooks).
# "validacion" = entreno solo con train, puedo medir nDCG.
# "entrega"    = entreno con train+test, uso todo el historial para predecir.
with open("./constantes/modo.txt") as f:
    MODO = f.read().strip()

assert MODO in ("validacion", "entrega"), f"MODO inválido: {MODO!r}"
print(f"MODO: {MODO}")

MODO: entrega


In [3]:
#b. Otras constantes.
%run "./constantes/constantes.ipynb"

In [4]:
#c. Verificación del modo (que los paths coincidan con lo que creo que estoy corriendo).
print(f"MODO: {MODO} | sufijo: {sufijo!r}")

MODO: entrega | sufijo: '_entrega'


#### 3. Funciones.

In [5]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [6]:
#a. Train.
# Fuerzo los ids a str: si el CSV los infiere como int, los merges contra
# df_libros / df_lectores devuelven NaN en silencio.
df_train = pd.read_csv(path_train_fe, dtype={"id_lector": str, "id_libro": str})

In [7]:
#b. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir, dtype={"id_lector": str})

In [8]:
#c. Libros y Lectores (lo tomo para armar la predicción).
df_libros = pd.read_csv(path_libros_fe, dtype={"id_libro": str})
df_lectores = pd.read_csv(path_lectores_fe, dtype={"id_lector": str})

In [9]:
#d. EASE^R entrenado en el notebook 3.
# No cargo ningun LGBMRanker: esta version usa EASE solo como predictor, para medir
# cuanto aporta (o cuanto degrada) la capa de contenido del ranker.
# Cargarlo en vez de reentrenarlo garantiza que sea exactamente el mismo objeto que
# se uso en el notebook 3, sin depender de que los parametros coincidan a mano.
B_ease = np.load(path_ease_B)
libros_idx_ease = joblib.load(path_ease_idx)
print(f"EASE cargado: {B_ease.shape} | {B_ease.nbytes / 1e9:.2f} GB | "
      f"{len(libros_idx_ease):,} items en el modelo")

EASE cargado: (26508, 26508) | 2.81 GB | 26,508 items en el modelo


#### 5. Preparación previa.

In [10]:
#a. No hay lista de features que armar ni assert contra el modelo.
# EASE^R no usa features: el score de un candidato sale de sumar las filas de B
# correspondientes a los libros que el lector ya leyo. Solo necesita el historial.
print("Predictor: EASE^R solo (sin LGBMRanker, sin feature engineering).")

Predictor: EASE^R solo (sin LGBMRanker, sin feature engineering).


#### 6. Predicción.

In [11]:
#a. Me aseguro de que todos los lectores a predecir tengan fila en df_lectores.
# Kaggle pide 832 lectores; si alguno no está en la tabla de lectores, el merge de
# caract_lector no matchea y sus features quedan en NaN. No los puedo descartar como a
# los libros sin metadata: hay que entregar recomendaciones para todos.
faltantes = set(df_a_predecir["id_lector"]) - set(df_lectores["id_lector"])
if faltantes:
    print(f"Agrego {len(faltantes)} lectores sin fila en df_lectores: {sorted(faltantes)}")
    df_lectores = pd.concat(
        [df_lectores, pd.DataFrame({"id_lector": sorted(faltantes)})],
        ignore_index=True
    )

Agrego 1 lectores sin fila en df_lectores: ['alpasil']


In [12]:
#a. Tablas de referencia para el feature engineering de los candidatos.
# Van primero porque el filtro del universo de libros las necesita.
# feature_engineering_test las toma como globales, así que los nombres del desempaquetado
# tienen que quedar exactamente así.
(media_global, caract_lector, caract_libros,
 afinidad_lector_autor_test, afinidad_lector_genero_test) = armar_tablas_referencia(
    df_train, df_libros, df_lectores
)

Nulos caract_lector: ninguno
Nulos caract_libros: {'autor': 78227, 'genero_libro_agrupado': 78220}
Duplicados (lector / libro / lector-autor / lector-genero): 0 0 0 0
Lectores: 11286 | Libros: 128743 | Libros sin interacciones en la base: 80676


In [13]:
#b. Universo de libros candidatos.
#i. Todos los libros con al menos una interacción.
conn = sqlite3.connect(path_db)
todos_los_libros = pd.read_sql("SELECT id_libro FROM interacciones", conn)["id_libro"].unique()
conn.close()

#ii. Filtro los libros sin la metadata que necesitan las features. Dos casos:
# - ~70 libros que están en interacciones pero no tienen fila en df_libros.
# - ~4 libros que sí están pero con autor en NaN: el merge de afinidad se hace sobre
#   ["id_lector", "autor"], y NaN nunca matchea contra NaN.
# Sin este filtro llegan al modelo con NaN y se rutean a un lado arbitrario del árbol,
# sin tirar error. Es el mismo arreglo que en el notebook 3.
n_antes = len(todos_los_libros)
libros_con_metadata = set(
    caract_libros.dropna(subset=["autor", "genero_libro_agrupado", "anio_edicion"])["id_libro"]
)
todos_los_libros = np.array([b for b in todos_los_libros if b in libros_con_metadata])

print(f"Universo de candidatos: {len(todos_los_libros):,} "
      f"(descarto {n_antes - len(todos_los_libros)} sin metadata)")

Universo de candidatos: 48,062 (descarto 75 sin metadata)


In [14]:
#c. Historial por lector (lo usa retrieval para excluir lo ya leído).
leidos_por_lector = (
    df_train[["id_lector", "id_libro"]]
    .groupby("id_lector")["id_libro"]
    .apply(set)
    .to_dict()
)

#d. Lectores a recomendarle.
id_lectores_predecir = df_a_predecir["id_lector"].unique()

In [15]:
#e. Verificación: los lectores a predecir, ¿tienen historial?
freq = df_train.groupby("id_lector").size()
cobertura = df_a_predecir["id_lector"].map(freq)
print(f"Modo: {MODO} | Filas de la base: {len(df_train):,}")
print("Lectores pedidos:", len(id_lectores_predecir))
print("Sin historial:", cobertura.isna().sum())
print("Mediana de frecuencia:", cobertura.median())

Modo: entrega | Filas de la base: 461,407
Lectores pedidos: 832
Sin historial: 320
Mediana de frecuencia: 95.0


In [16]:
#f. Calculamos ranking final para producción.
#i. Lista donde almacenaremos las recomendaciones.
recomendaciones = []
sin_candidatos = []
total_lectores = len(id_lectores_predecir)

print("Comienza la predicción general.")

#ii. Recorro cada id_lector.
for i, id_lector in enumerate(id_lectores_predecir, start=1):
    if i % 50 == 0 or i == 1:
        print("Lector {}/{}".format(i, total_lectores))

    #1. Retrieval.
    libros_candidatos_a_recomendar = retrieval(id_lector)

    # Si no tiene candidatos queda fuera del entregable: lo registro en vez de saltearlo
    # en silencio, porque Kaggle promedia sobre TODOS los lectores pedidos y un lector
    # faltante suma un 0 al promedio.
    if len(libros_candidatos_a_recomendar) == 0:
        sin_candidatos.append(id_lector)
        continue

    #2. Score colaborativo directo de EASE^R.
    # No hace falta feature_engineering_test: EASE solo necesita el historial del lector.
    # Eso hace el loop mucho mas rapido (sin merges sobre ~44.000 candidatos por lector).
    df_features_candidatos = pd.DataFrame({
        "id_lector": id_lector,
        "id_libro": libros_candidatos_a_recomendar
    })
    df_features_candidatos["score_ranking"] = score_ease(
        id_lector, libros_candidatos_a_recomendar,
        B_ease, libros_idx_ease, leidos_por_lector
    )

    #3. Ordeno de mayor a menor y me quedo con los 20 mejores.
    top_20 = df_features_candidatos.sort_values("score_ranking", ascending=False).head(20)

    #4. Guardo las recomendaciones (el ORDEN de las filas es la métrica: nDCG mide posición).
    recomendaciones.append(top_20[["id_lector", "id_libro"]])

#iii. Uno todas las recomendaciones.
df_recomendaciones = pd.concat(recomendaciones, ignore_index=True)

print("\nPredicción general finalizada.")
print("Cantidad de recomendaciones:", len(df_recomendaciones))
if sin_candidatos:
    print(f"ATENCIÓN: {len(sin_candidatos)} lectores sin candidatos, quedan fuera:", sin_candidatos[:10])

#iv. Cuantos lectores no tienen historial dentro del modelo de EASE: para esos el score
# es todo cero y el top-20 queda arbitrario. Es el punto ciego del colaborativo puro.
sin_senal = sum(
    1 for l in id_lectores_predecir
    if not any(b in libros_idx_ease for b in leidos_por_lector.get(l, set()))
)
print(f"Lectores sin ningún libro dentro del modelo de EASE: {sin_senal} de {total_lectores}")

Comienza la predicción general.
Lector 1/832
Lector 50/832
Lector 100/832
Lector 150/832
Lector 200/832
Lector 250/832
Lector 300/832
Lector 350/832
Lector 400/832
Lector 450/832
Lector 500/832
Lector 550/832
Lector 600/832
Lector 650/832
Lector 700/832
Lector 750/832
Lector 800/832

Predicción general finalizada.
Cantidad de recomendaciones: 16640
Lectores sin ningún libro dentro del modelo de EASE: 16 de 832


#### 7. Exportación de las recomendaciones.

In [17]:
#a. Comprobaciones antes de exportar.
assert df_recomendaciones["id_lector"].nunique() == len(id_lectores_predecir), (
    f"Faltan lectores: entrego {df_recomendaciones['id_lector'].nunique()} "
    f"de {len(id_lectores_predecir)}"
)
assert (df_recomendaciones.groupby("id_lector").size() == 20).all(), (
    "Hay lectores con distinta cantidad de 20 recomendaciones"
)
assert df_recomendaciones.duplicated(["id_lector", "id_libro"]).sum() == 0, (
    "Hay pares (lector, libro) duplicados"
)
print("Lectores entregados:", df_recomendaciones["id_lector"].nunique())
print("Filas por lector:", df_recomendaciones.groupby("id_lector").size().unique())
print("Comprobaciones: ok")

Lectores entregados: 832
Filas por lector: [20]
Comprobaciones: ok


In [18]:
#b. Defino el número de versión (a mano).
version = "18"

In [19]:
#c. Exporto la versión de esta corrida.
# index=False y sin reordenar: el orden de las filas dentro de cada lector ES el ranking.
ruta = f"./outputs/entregable_{version}.csv"
df_recomendaciones.to_csv(ruta, index=False)
print(f"Exportado: {ruta} ({len(df_recomendaciones):,} filas)")

Exportado: ./outputs/entregable_18.csv (16,640 filas)


In [20]:
#d. Compruebo el archivo escrito (releo, por las dudas).
chequeo = pd.read_csv(ruta, dtype={"id_lector": str, "id_libro": str})
print("Lectores pedidos:   ", df_a_predecir["id_lector"].nunique())
print("Lectores entregados:", chequeo["id_lector"].nunique())
print(chequeo.groupby("id_lector").size().value_counts())
print("\nPrimeras filas:")
print(chequeo.head(5))

Lectores pedidos:    832
Lectores entregados: 832
20    832
Name: count, dtype: int64

Primeras filas:
    id_lector                  id_libro
0  05-03-1970  la-conjura-de-los-necios
1  05-03-1970         el-viejo-y-el-mar
2  05-03-1970     rebelion-en-la-granja
3  05-03-1970                el-jugador
4  05-03-1970                siddhartha
